# AI Pullback Prediction — Kaggle GPU

Notebook untuk menjalankan training LSTM/Transformer XAUUSD di Kaggle. Aktifkan **Settings → Accelerator → GPU** dan tambahkan dataset Parquet hasil `prepare-data` ke notebook sebagai Kaggle Input.

Data wajib berisi bid/ask OHLC dan timestamp sebagai DatetimeIndex (format Parquet proyek). Training memakai 2015–2020; 2021 early stopping; 2022 kalibrasi; 2023 evaluasi. 2024+ tidak disentuh.

Jalankan sel dari atas ke bawah. Hasil disimpan di `/kaggle/working/artifacts/`.


In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

REPO_URL = "https://github.com/NodePro-gif/AI-Pullback-Prediction-.git"
REPO_DIR = Path('/kaggle/working/AI-Pullback-Prediction-')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[deep-learning]'], cwd=REPO_DIR, check=True)
print('Repository and deep-learning dependencies installed:', REPO_DIR)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak terdeteksi. Kaggle: Settings → Accelerator → GPU, lalu restart session.')
print('GPU:', torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1024**3:.1f} GiB')
print('CUDA:', torch.version.cuda)
torch.backends.cudnn.benchmark = True


## Pilih dataset dan konfigurasi

`DATA_PATH` boleh diisi path Parquet di `/kaggle/input/...`. Jika dikosongkan, notebook mencari file `.parquet` di `/kaggle/input` dan berhenti jika hasilnya ambigu.


In [ ]:
# EDIT jika dataset memiliki lebih dari satu file Parquet:
DATA_PATH = ''  # contoh: '/kaggle/input/xauusd-data/XAUUSD_M1.parquet'
TIMEFRAME = 'M1'  # M1 atau M5
ARCHITECTURE = 'lstm'  # lstm atau transformer
SEQUENCE_LENGTH = 128
EPOCHS = 30
BATCH_SIZE = 256  # naikkan bertahap jika VRAM masih cukup
HIDDEN_DIM = 96
NUM_LAYERS = 2
LEARNING_RATE = 1e-3
PATIENCE = 5

if not DATA_PATH:
    candidates = sorted(glob.glob('/kaggle/input/**/*.parquet', recursive=True))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
    elif not candidates:
        raise FileNotFoundError('Tidak menemukan Parquet. Tambahkan dataset Parquet XAUUSD melalui Add Input.')
    else:
        print('\n'.join(candidates))
        raise RuntimeError('Ada beberapa Parquet. Isi DATA_PATH dengan file yang benar lalu jalankan ulang sel ini.')
DATA_PATH = str(Path(DATA_PATH))
if not Path(DATA_PATH).is_file():
    raise FileNotFoundError(DATA_PATH)
if TIMEFRAME not in {'M1', 'M5'} or ARCHITECTURE not in {'lstm', 'transformer'}:
    raise ValueError('TIMEFRAME harus M1/M5 dan ARCHITECTURE harus lstm/transformer')
print('Dataset:', DATA_PATH)


In [ ]:
ARTIFACT_DIR = Path('/kaggle/working/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / f'xauusd_{TIMEFRAME.lower()}_{ARCHITECTURE}.joblib'
cmd = [
    sys.executable, '-m', 'pullback_prediction.cli', 'train-sequence',
    '--input', DATA_PATH,
    '--output', str(MODEL_PATH),
    '--architecture', ARCHITECTURE,
    '--timeframe', TIMEFRAME,
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-dim', str(HIDDEN_DIM),
    '--num-layers', str(NUM_LAYERS),
    '--learning-rate', str(LEARNING_RATE),
    '--patience', str(PATIENCE),
    '--device', 'cuda',
    '--mixed-precision',
    '--num-workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('Training finished.')


In [ ]:
metrics_path = MODEL_PATH.with_suffix('.metrics.json')
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))
print('\nArtifacts:')
for p in sorted(ARTIFACT_DIR.glob('*')):
    print(p.name, f'{p.stat().st_size / 1024**2:.2f} MiB')


## Unduh hasil

Di panel Files Kaggle, buka `/kaggle/working/artifacts/`. Simpan bersama-sama file `.joblib`, `.pt`, dan `.metrics.json`; metadata model membutuhkan checkpoint `.pt` saat inference.

Catatan: notebook ini menyiapkan dan menjalankan training, bukan menjamin kualitas profit. Performa GPU, kualitas prediksi, dan backtest pada data riil tetap harus dievaluasi.
